[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/en_clase/S05_05_contrastes_causalidad_en_clase.ipynb)

# S05_05 · Estadística con Python: ¿los festivos suben la ocupación? ¿y el calor la baja? (en clase)

<h2 align="center">Comprobar una diferencia con un contraste y no confundir correlación con causalidad</h2>

| | |
|---|---|
| ⏱ **Cuándo** | Antes de empezar el reto · unos 40 min |
| 👥 **Cómo** | En parejas: una persona escribe, la otra lee y comprueba (cambiad de papel al empezar la parte B) |
| 🧰 **Usáis** | 🌐 Google Colab (este cuaderno) · no hay que instalar nada |
| 🏁 **Al terminar** | Ejercicio 1 (contraste de hipótesis) y ejercicio 5 (controlar una variable de confusión) con todo ✅ en el paso 6 |

**Este cuaderno es la base del reto del sprint** («¿La ocupación depende de esto?»): en él se responde a las mismas preguntas con datos de TurisData.

**Este cuaderno tiene dos partes:**

- **1 · Guía paso a paso**: solo lo que hay que hacer. Va en dos bloques: **A · Contraste de hipótesis** y **B · Correlación y causalidad**.
- **2 · Explicaciones**: el porqué de cada cosa, para cuando lo necesitéis (está al final, también en dos bloques: A y B).
- Si queréis, seguid solo la parte 1.

**Solo trae los dos ejercicios de clase**: el **1 del contraste de hipótesis** (parte A) y el **5 de la confusión** (parte B). Los demás (2, 3, 4, 6 y el mini-reto) están en el cuaderno completo [`S05_05_contrastes_causalidad`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_05_contrastes_causalidad.ipynb), para casa.

## Cómo se usa este cuaderno (leedlo una vez)

**Un cuaderno es una mezcla de textos y celdas de código.** Las celdas de código tienen a la izquierda un botón **▶**.

1. **Ejecutar una celda**
   - Pulsad el botón **▶** de la izquierda, o haced clic dentro de la celda y pulsad **Mayús + Intro**.
   - Debajo de la celda aparece **lo que ha salido**.
2. **De arriba abajo, en orden**
   - Cada celda usa lo que han hecho las anteriores.
   - Si os saltáis una, saldrá un error rojo que dice `NameError`. No pasa nada: ejecutad las que faltan y repetid.
3. **Las celdas con `...`**
   - Son los ejercicios. Los **tres puntos `...` se sustituyen** por vuestro código.
   - No borréis el resto de la celda.
4. **Cada paso termina en ✅**
   - Es lo que **tiene que salir**.
   - Los decimales lejanos pueden cambiar un poco en vuestro ordenador; lo demás tiene que ser **igual**.
   - Si no sale, no sigáis: mirad la parte 2, apartado «Si os atascáis».
5. **Si Colab se reinicia o cerráis la pestaña**
   - Se pierde lo ejecutado, no lo escrito.
   - Volved a ejecutar las celdas **desde el paso 0 hasta donde estabais**.

# 1 · Guía paso a paso

**Qué vais a hacer**

- **Parte A · Contraste de hipótesis (ejercicio 1):** comprobar si la ocupación de los **días festivos** es distinta de la de los días normales, o si la diferencia puede ser casualidad.
- **Parte B · Correlación y causalidad (ejercicio 5):** ver que el calor **parece** bajar la ocupación y descubrir que, en gran parte, es un **espejismo** causado por la temporada.
- Son dos preguntas del cliente: «¿los festivos suben la ocupación?» y «¿hace bajar la ocupación el calor?».

## Antes de empezar · Paso 0 · Preparar el cuaderno (3 min)

| # | Qué hacéis |
|---|---|
| **0.1** | Ejecutad la celda de abajo (la de «Preparación»). Tarda unos segundos: busca el fichero de datos y, si no está, lo descarga. |
| **0.2** | Ejecutad la celda siguiente (la de las librerías). |

✅ Con la primera sale `Preparación lista ✅` y con la segunda `Librerías listas ✅`.

*No hace falta entender estas dos celdas: dejan preparado lo que usaremos después.*

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

RESULTADOS = {}

def comprobar(nombre, condicion, ayuda):
    """Escribe ✅ si la condición es cierta y ❌ con una pista si no. No hace falta entender cómo funciona."""
    try:
        ok = bool(condicion())
    except Exception:
        ok = False
    RESULTADOS[nombre] = ok
    print(("✅ " + nombre) if ok else ("❌ " + nombre + " → " + ayuda))

dato("ocupacion_diaria.csv")
print("Preparación lista ✅")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

print("Librerías listas ✅")

# Parte A · Contraste de hipótesis (ejercicio 1) · unos 15 min

**Qué vais a hacer**

- Tenéis la ocupación de **1096 días**. Solo **61** son festivos y los otros **1035** son días normales.
- Los festivos tienen la ocupación más alta, pero... ¿es una **diferencia de verdad** o podría ser **casualidad**?
- Un **contraste de hipótesis** responde con un número: el **p-valor**.
- Primero veréis cómo funciona con dos grupos pequeños inventados (paso 2) y después lo haréis con los datos reales: es el **ejercicio 1**.

## Paso 1 · Cargar los días y separarlos en dos grupos (4 min)

**Los datos**

- Una **fila por día**, con las columnas `fecha`, `ocupacion_pct`, `temperatura_c`, `festivo` (1 si es festivo y 0 si no) y `precio_medio`.
- Añadimos una columna más, `mes` (del 1 al 12), que usaremos en la parte B.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **1.1** | Ejecutad la celda 1.1 (carga el fichero en una tabla llamada `diaria` y añade la columna `mes`). | `(1096, 6)` |
| **1.2** | Ejecutad la celda 1.2 (separa la ocupación de los festivos y la de los días normales). | `Festivos: 61 días, media 83.9` y `Normales: 1035 días, media 67.6` |

- `(1096, 6)` quiere decir: **1096 filas** (días) y **6 columnas**.
- Las dos medias ya se ven distintas: **83.9 % frente a 67.6 %**. La pregunta es si esa diferencia es de fiar.

In [ ]:
# 1.1 · Cargar la tabla y añadir el mes
diaria = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"])
diaria["mes"] = diaria["fecha"].dt.month
print(diaria.shape)

In [ ]:
# 1.2 · Dos grupos: la ocupación de los festivos y la de los días normales
festivos = diaria.loc[diaria["festivo"] == 1, "ocupacion_pct"]
normales = diaria.loc[diaria["festivo"] == 0, "ocupacion_pct"]
print(f"Festivos: {len(festivos)} días, media {festivos.mean():.1f}")
print(f"Normales: {len(normales)} días, media {normales.mean():.1f}")

## Paso 2 · Ejercicio 1 · ¿Los festivos suben la ocupación? (11 min) 🏫

**Objetivo**

- Aplicar el contraste **t de Student** a las dos series y guardar tres resultados: el **p-valor**, la **diferencia de medias** y si se **rechaza la hipótesis nula**.
- **Hipótesis nula (H0):** «no hay diferencia entre festivos y días normales».
- **Regla:** si el p-valor es **menor que 0.05**, se rechaza H0: la diferencia es **demasiado grande para ser casualidad**.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **2.1** | **A mano, con dos grupos de 5 números.** Ejecutad la celda 2.1: compara un caso con diferencia y otro sin ella. | `Con diferencia: p = 0.004 → se rechaza H0` y `Sin diferencia: p = 1.0 → no se rechaza H0` |
| **2.2** | **Vuestro turno.** En la celda 2.2, sustituid cada `...` por la misma cuenta pero con `festivos` y `normales`. | No sale nada |
| **2.3** | Ejecutad la celda 2.3 (la comprobación). | Cuatro líneas con ✅ |
| **2.4** | Ejecutad la celda 2.4 (muestra el resultado y dibuja las dos distribuciones). | `Diferencia: 16.3 puntos · p = 1.1e-30 → se rechaza H0` y un gráfico con dos grupos de barras |

- **Pistas para 2.2:**
  - `resultado`: `stats.ttest_ind(festivos, normales, equal_var=False)`
  - `p_festivo`: `resultado.pvalue`
  - `dif_medias`: `festivos.mean() - normales.mean()`
  - `rechazar_h0`: `p_festivo < 0.05`
- `equal_var=False` es la variante de **Welch**: no supone que los dos grupos varíen igual. Es la más prudente y la que usaremos siempre.
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.

In [ ]:
# 2.1 · A mano, con dos grupos pequeños
a = [80, 85, 90, 75, 95]                      # media 85
b = [60, 65, 70, 55, 75]                      # media 65: 20 puntos menos
p_con = stats.ttest_ind(a, b, equal_var=False).pvalue
print(f"Con diferencia: p = {p_con:.3f} → " + ("se rechaza H0" if p_con < 0.05 else "no se rechaza H0"))

c = [70, 72, 68, 71, 69]                      # media 70
d = [69, 71, 67, 70, 73]                      # media 70: igual que c
p_sin = stats.ttest_ind(c, d, equal_var=False).pvalue
print(f"Sin diferencia: p = {p_sin:.1f} → " + ("se rechaza H0" if p_sin < 0.05 else "no se rechaza H0"))

In [ ]:
# 2.2 · TU TURNO: lo mismo que en la celda 2.1, pero con festivos y normales
resultado = ...        # stats.ttest_ind(festivos, normales, equal_var=False)
p_festivo = ...        # resultado.pvalue
dif_medias = ...       # festivos.mean() - normales.mean()
rechazar_h0 = ...      # p_festivo < 0.05

In [ ]:
# 2.3 · Comprobación del ejercicio 1 (no hay que cambiar nada)
comprobar("t de Student: estadístico", lambda: abs(resultado.statistic - 19.15) < 0.05, "el estadístico t debe salir 19.15: stats.ttest_ind(festivos, normales, equal_var=False). ¿Has cambiado los puntos ... por tu código?")
comprobar("t de Student: p-valor", lambda: p_festivo < 1e-20, "el p-valor debe ser minúsculo (≈1e-30): resultado.pvalue")
comprobar("t de Student: diferencia", lambda: abs(dif_medias - 16.30) < 0.01, "la diferencia debe salir 16.30: festivos.mean() - normales.mean() (festivos primero)")
comprobar("t de Student: decisión", lambda: isinstance(rechazar_h0, (bool, np.bool_)) and rechazar_h0 == True, "debe valer True: p_festivo < 0.05")

In [ ]:
# 2.4 · Mirar el resultado
print(f"Diferencia: {dif_medias:.1f} puntos · p = {p_festivo:.1e} → " + ("se rechaza H0" if rechazar_h0 else "no se rechaza H0"))
plt.figure(figsize=(7, 3.5))
plt.hist(normales, bins=20, alpha=0.7, density=True, color="lightsteelblue", label=f"Normales (media {normales.mean():.1f})")
plt.hist(festivos, bins=12, alpha=0.7, density=True, color="darkorange", label=f"Festivos (media {festivos.mean():.1f})")
plt.xlabel("Ocupación diaria (%)")
plt.ylabel("Proporción de días")
plt.title("Los festivos están casi todos por encima de los días normales")
plt.legend()
plt.tight_layout()
plt.show()

✅ Tiene que salir: `✅ t de Student: estadístico`, `✅ t de Student: p-valor`, `✅ t de Student: diferencia` y `✅ t de Student: decisión`. Si falta alguno, no sigáis.

# Parte B · Correlación y causalidad (ejercicio 5) · unos 25 min

*Cambiad de papel: quien escribía, ahora lee y comprueba.*

**Qué vais a hacer**

- Veréis que cuando hace más calor, **baja la ocupación**: una **correlación negativa** de −0.63.
- ¿Habría que «enfriar» los hoteles para llenarlos? **No.** Que dos cosas vayan juntas **no quiere decir que una cause la otra**.
- Con un **modelo de regresión** vais a **quitar el efecto del mes** y comprobar qué queda del calor: es el **ejercicio 5**.

## Paso 3 · La correlación y la primera regresión (6 min)

**Objetivo**

- Medir cuánto se relacionan la **temperatura** y la **ocupación**.
- Ajustar un primer modelo que dice «cuántos puntos baja la ocupación por cada grado».

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **3.1** | Ejecutad la celda 3.1 (calcula la correlación de Pearson y dibuja una nube de puntos). | `Correlación: -0.63` y un gráfico de puntos que **bajan** hacia la derecha |
| **3.2** | Ejecutad la celda 3.2 (ajusta `modelo_simple`: la ocupación según la temperatura). | `Ocupación = 107.73 + (-1.86) × temperatura` |

- **La correlación** va de −1 a 1: **−0.63** es negativa y bastante fuerte (cuanto más calor, menos ocupación).
- **El modelo simple** dice: «por cada grado más, **−1.86 puntos** de ocupación». Parece un efecto del calor... pero falta mirar algo.

In [ ]:
# 3.1 · Correlación entre temperatura y ocupación, y gráfico
print("Correlación:", round(diaria["ocupacion_pct"].corr(diaria["temperatura_c"]), 2))
plt.figure(figsize=(7, 3.5))
plt.scatter(diaria["temperatura_c"], diaria["ocupacion_pct"], s=8, color="steelblue")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Ocupación (%)")
plt.title("A más calor, menos ocupación")
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 · Modelo simple: la ocupación según la temperatura (ya hecho)
modelo_simple = smf.ols("ocupacion_pct ~ temperatura_c", data=diaria).fit()
print(f"Ocupación = {modelo_simple.params['Intercept']:.2f} + ({modelo_simple.params['temperatura_c']:.2f}) × temperatura")

## Paso 4 · Mirar de dónde sale el espejismo (6 min)

**Objetivo**

- Ver cómo se lee un modelo de regresión con un ejemplo muy sencillo.
- Descubrir que el **mes** influye a la vez en la temperatura y en la ocupación.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **4.1** | Ejecutad la celda 4.1 (un modelo con solo `festivo`: **cómo se leen los números**). | `Intercept = 67.58` y `festivo = 16.3` |
| **4.2** | Ejecutad la celda 4.2 (temperatura y ocupación **medias de cada mes**). | Una tabla de 12 filas: enero `17.1` y `78.8`, julio `25.2` y `63.0` |

- **En 4.1:**
  - `Intercept` es el valor de partida: la media de los días **normales** (67.58).
  - El número de `festivo` es **cuánto se suma** cuando el día es festivo: **+16.3**. Es **justo** la diferencia de medias del ejercicio 1.
  - Así se lee cada número de un modelo: «cuánto cambia la ocupación cuando esa variable sube una unidad».
- **En 4.2:**
  - En **enero** (17 °C) la ocupación es del **78.8 %**. En **julio** (25 °C) es del **63.0 %**.
  - **El invierno** es más fresco y, en Canarias, es **temporada alta**: llega gente que huye del frío de Europa.
  - Por eso la ocupación parece bajar con el calor: **el mes** influye en las dos cosas a la vez. Se llama **variable de confusión**.

In [ ]:
# 4.1 · Un modelo con solo el festivo: cómo se leen los coeficientes
modelo_festivo = smf.ols("ocupacion_pct ~ festivo", data=diaria).fit()
print(f"Intercept = {modelo_festivo.params['Intercept']:.2f}")
print(f"festivo = {modelo_festivo.params['festivo']:.1f}")

In [ ]:
# 4.2 · La temperatura y la ocupación medias de cada mes
por_mes = diaria.groupby("mes")[["temperatura_c", "ocupacion_pct"]].mean().round(1)
print(por_mes)

## Paso 5 · Ejercicio 5 · Quitar el efecto del mes (10 min) 🏫

**Objetivo**

- Ajustar un modelo que **tenga en cuenta el mes y el festivo** y ver qué pasa con el efecto de la temperatura.
- La fórmula es la misma de 3.2 y se le **añaden** `festivo` y `C(mes)`. La `C(...)` le dice que el mes es una **categoría** (12 meses distintos), no un número.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **5.1** | **Vuestro turno.** En la celda 5.1, sustituid los `...` por el modelo y por los tres números que se piden. | No sale nada |
| **5.2** | Ejecutad la celda 5.2 (la comprobación). | Cuatro líneas con ✅ |
| **5.3** | Ejecutad la celda 5.3 (muestra el resultado: antes y después de quitar el efecto del mes). | `Sin tener en cuenta el mes: -1.86 puntos por grado` y `Teniendo en cuenta el mes: -0.09 puntos por grado (p = 0.48)` |

- **Pistas para 5.1:**
  - `modelo_control`: `smf.ols("ocupacion_pct ~ temperatura_c + festivo + C(mes)", data=diaria).fit()`
  - `coef_temp_simple`: `modelo_simple.params["temperatura_c"]`
  - `coef_temp_control`: `modelo_control.params["temperatura_c"]`
  - `p_temp_control`: `modelo_control.pvalues["temperatura_c"]`
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.
- **El resultado de 5.3 dice:** al quitar el efecto del mes, el calor **casi no tiene efecto** (−0.09) y ese efecto **ya no es significativo** (p = 0.48). **El festivo, en cambio, sigue subiendo la ocupación** (+9 puntos).

In [ ]:
# 5.1 · TU TURNO: el modelo que tiene en cuenta el festivo y el mes
modelo_control = ...         # smf.ols("ocupacion_pct ~ temperatura_c + festivo + C(mes)", data=diaria).fit()
coef_temp_simple = ...       # el coeficiente de la temperatura del modelo simple: modelo_simple.params["temperatura_c"]
coef_temp_control = ...      # el de la temperatura del modelo de control: modelo_control.params["temperatura_c"]
p_temp_control = ...         # su p-valor: modelo_control.pvalues["temperatura_c"]

In [ ]:
# 5.2 · Comprobación del ejercicio 5 (no hay que cambiar nada)
comprobar("Confusión: modelo", lambda: "C(mes)" in modelo_control.model.formula and "festivo" in modelo_control.model.formula, "la fórmula debe ser \"ocupacion_pct ~ temperatura_c + festivo + C(mes)\" y terminar en .fit(). ¿Has cambiado los puntos ... por tu código?")
comprobar("Confusión: coeficiente simple", lambda: abs(coef_temp_simple + 1.86) < 0.01, "debe salir -1.86: modelo_simple.params[\"temperatura_c\"]")
comprobar("Confusión: coeficiente con control", lambda: abs(coef_temp_control) < 0.3, "debe quedar cerca de 0 (≈ -0.09): modelo_control.params[\"temperatura_c\"]. ¿Has usado el modelo de control y no el simple?")
comprobar("Confusión: p-valor", lambda: 0.05 < p_temp_control < 1, "debe ser mayor que 0.05 (≈ 0.48): modelo_control.pvalues[\"temperatura_c\"]")

In [ ]:
# 5.3 · Mirar el resultado
print(f"Sin tener en cuenta el mes: {coef_temp_simple:.2f} puntos por grado")
print(f"Teniendo en cuenta el mes: {coef_temp_control:.2f} puntos por grado (p = {p_temp_control:.2f})")
print(f"El efecto del festivo se mantiene: +{modelo_control.params['festivo']:.1f} puntos (p = {modelo_control.pvalues['festivo']:.0e})")

## Paso 6 · Cerrar los dos ejercicios (2 min)

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **6.1** | Ejecutad la celda 6.1 (cuenta las comprobaciones ✅ de cada ejercicio por separado). | `Parte A · Contraste de hipótesis (ejercicio 1): 4 de 4 ✅`, `Parte B · Correlación y causalidad (ejercicio 5): 4 de 4 ✅` y `🎉 Los ejercicios 1 y 5 están hechos` |
| **6.2** | Escribid con vuestras palabras la **conclusión para el cliente** (2 frases, en un papel o en el chat de la pareja): qué pasa con los festivos y qué pasa con el calor. | Algo como: «los festivos suben la ocupación; el calor **parece** bajarla solo porque coincide con la temporada» |
| **6.3** | Si os sobra tiempo, abrid el cuaderno completo [`S05_05_contrastes_causalidad`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_05_contrastes_causalidad.ipynb) y probad los ejercicios 2 (tamaño del efecto) y 3 (chi-cuadrado). | — |

In [ ]:
# 6.1 · Resumen por ejercicio
N1 = ["t de Student: estadístico", "t de Student: p-valor", "t de Student: diferencia", "t de Student: decisión"]
N5 = ["Confusión: modelo", "Confusión: coeficiente simple", "Confusión: coeficiente con control", "Confusión: p-valor"]
bien1 = sum(RESULTADOS.get(n, False) for n in N1)
bien5 = sum(RESULTADOS.get(n, False) for n in N5)
print(f"Parte A · Contraste de hipótesis (ejercicio 1): {bien1} de 4 ✅")
print(f"Parte B · Correlación y causalidad (ejercicio 5): {bien5} de 4 ✅")
if bien1 == 4 and bien5 == 4:
    print("🎉 Los ejercicios 1 y 5 están hechos")
else:
    print("Faltan comprobaciones: mirad las líneas ❌ de las celdas 2.3 (ejercicio 1) y 5.2 (ejercicio 5)")

## ✅ Terminado cuando

**Parte A · Contraste de hipótesis (ejercicio 1)**

- [ ] La celda 2.3 muestra las cuatro líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **qué significa un p-valor muy pequeño**.

**Parte B · Correlación y causalidad (ejercicio 5)**

- [ ] La celda 5.2 muestra las cuatro líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **por qué el calor parecía bajar la ocupación y no la baja**.

**Los dos**

- [ ] La celda 6.1 dice `🎉 Los ejercicios 1 y 5 están hechos`.

---

# 2 · Explicaciones

*Para cuando una cosa no sale o queréis entender por qué se hace así. No hace falta leerlo para resolver los ejercicios. Está en dos bloques, igual que la guía: **A · Contraste de hipótesis** y **B · Correlación y causalidad**.*

# Explicaciones de la parte A · Contraste de hipótesis (ejercicio 1)

## La idea: ¿diferencia de verdad o casualidad?

- Si miramos 1096 días, los festivos tienen de media **83.9 %** y los normales **67.6 %**. Hay una diferencia de **16.3 puntos**.
- Pero con cualquier grupo de datos, **dos medias nunca salen exactamente iguales** solo por azar.
- La pregunta es: **¿esta diferencia es tan grande que no se explica por casualidad?**
- Un **contraste de hipótesis** contesta con un procedimiento fijo:
  1. **Hipótesis nula (H0):** «no hay diferencia» (los festivos y los días normales tienen la misma ocupación media).
  2. Se calcula un **estadístico** (aquí, la `t`) y su **p-valor**.
  3. Se **decide**: si el p-valor es menor que **0.05**, se rechaza H0.

## Qué es el p-valor

- **El p-valor** es la probabilidad de ver una diferencia **tan grande como la observada (o más) si H0 fuera cierta**, es decir, si en realidad no hubiera diferencia.
- Se lee así:

| p-valor | Se lee | Decisión |
|---|---|---|
| **Pequeño** (menor que 0.05) | «Si no hubiera diferencia, esto sería muy raro» | Se **rechaza H0**: la diferencia es real |
| **Grande** (mayor que 0.05) | «Esto podría pasar por casualidad» | **No se rechaza H0**: no hay pruebas de diferencia |

- **En los ejemplos de 2.1:**
  - Los grupos `a` y `b` tienen medias de 85 y 65: p = **0.004**, menor que 0.05 → **se rechaza H0**.
  - Los grupos `c` y `d` tienen la misma media (70): p = **1.0** → **no se rechaza H0**.
- **En los datos reales:** p = **1.1e-30**, es decir, `0.0000…0011` con treinta ceros. La casualidad **no explica** la diferencia entre festivos y normales.
- **Cuidado con dos errores muy comunes:**
  - El p-valor **no** es la probabilidad de que H0 sea cierta.
  - Un p-valor pequeño dice que la diferencia **existe**, no que sea **grande** ni **importante**. Para eso se mira la propia diferencia (16.3 puntos).

## Las piezas de la celda 2.2

| Variable | Qué es | Cómo se calcula | Con los datos |
|---|---|---|---|
| **`resultado`** | Lo que devuelve el contraste: el estadístico `t` y el p-valor | `stats.ttest_ind(festivos, normales, equal_var=False)` | `t = 19.15` |
| **`p_festivo`** | El p-valor | `resultado.pvalue` | 1.1e-30 |
| **`dif_medias`** | Cuánto más alta es la ocupación en festivo | `festivos.mean() - normales.mean()` | 16.3 puntos |
| **`rechazar_h0`** | La decisión: `True` si el p-valor es menor que 0.05 | `p_festivo < 0.05` | `True` |

- **`ttest_ind`:** compara las medias de **dos grupos independientes** (un día es festivo o es normal, no las dos cosas). La `t` mide cuántas veces cabe el error en la diferencia de medias: cuanto mayor, más clara es la diferencia.
- **`equal_var=False`:** es la versión de **Welch**. No supone que los dos grupos varíen igual (aquí hay 61 festivos y 1035 normales). Es la opción prudente.
- **`rechazar_h0`** vale `True` o `False` (un valor **booleano**). Se obtiene al comparar con `<`.

## Las palabras de la parte A

| Palabra | Qué significa |
|---|---|
| **Hipótesis nula (H0)** | La suposición de que «no hay diferencia» |
| **Contraste de hipótesis** | Un procedimiento para decidir si H0 se rechaza o no |
| **t de Student (de Welch)** | El contraste que compara las medias de dos grupos |
| **Estadístico (`t`)** | El número que resume lo grande que es la diferencia frente al azar |
| **p-valor** | La probabilidad de ver una diferencia así si H0 fuera cierta |
| **α = 0.05** | El umbral habitual: por debajo, se rechaza H0 |
| **Booleano** | Un valor que solo puede ser `True` o `False` |
| **`.loc[condición, columna]`** | Elige las filas que cumplen la condición y una columna |
| **✅ / ❌** | Lo que escribe la función `comprobar`: ✅ está bien; ❌ hay algo que corregir (con una pista) |

## Por qué se hace así

- **¿Por qué 0.05?**
  - Es una convención: aceptamos equivocarnos como mucho 1 de cada 20 veces. Se podría usar otro valor, pero este es el más habitual.
- **¿Por qué «no se rechaza H0» y no «H0 es cierta»?**
  - Que no veamos una diferencia no prueba que no exista: puede que haya pocos datos. Por eso se dice que **no hay pruebas**.
- **¿Por qué `festivos` antes que `normales` en la resta?**
  - Para que el signo tenga sentido: **+16.3** quiere decir «los festivos tienen más». Si se invierte, sale −16.3.
- **¿Por qué `p_festivo < 0.05` da `True` y no un texto?**
  - Una comparación siempre devuelve `True` o `False`. Es lo que necesita la comprobación.

# Explicaciones de la parte B · Correlación y causalidad (ejercicio 5)

## La idea: ir juntas no es causarse

- **Correlación:** dos variables **suben o bajan a la vez**. Se mide de −1 a 1.
  - **Cerca de 1:** suben juntas. **Cerca de −1:** una sube cuando la otra baja. **Cerca de 0:** no hay relación en línea recta.
  - La de temperatura y ocupación es **−0.63**: cuanto más calor, menos ocupación.
- **Causalidad:** una variable **provoca** el cambio de la otra.
- La correlación **no** demuestra causalidad. Un ejemplo: los helados y las quemaduras de sol suben a la vez, pero no es que los helados quemen: **el verano** provoca las dos cosas.
- **Aquí pasa lo mismo:** el **mes** provoca dos cosas a la vez. En invierno hace menos calor y hay **más turistas**, así que parece que el frío llena los hoteles.
- **Variable de confusión:** una variable (el mes) que influye en las otras dos y crea una relación que no es un efecto directo.

## Cómo se lee un modelo de regresión

- Una **regresión** busca la recta (o el plano) que mejor se ajusta a los datos. Se escribe con una **fórmula**:

| Fórmula | Se lee |
|---|---|
| `"ocupacion_pct ~ temperatura_c"` | «La ocupación **depende de** la temperatura» |
| `"ocupacion_pct ~ festivo"` | «La ocupación depende de si es festivo» |
| `"ocupacion_pct ~ temperatura_c + festivo + C(mes)"` | «Depende de la temperatura, del festivo **y** del mes, **a la vez**» |

- **El coeficiente de cada variable** (`.params["temperatura_c"]`) es **cuánto cambia la ocupación cuando esa variable sube una unidad, manteniendo las demás iguales**.
- **El p-valor de cada coeficiente** (`.pvalues["temperatura_c"]`) dice si ese efecto se puede distinguir de cero (menor que 0.05) o no (mayor).
- **`C(mes)`** le dice que el mes es una **categoría**: crea una «etiqueta» para cada uno de los 12 meses, para que el modelo pueda corregir **cada mes por separado**.
- **El truco de «tener en cuenta el mes»:** el modelo compara días **del mismo mes**. Dentro de un mismo mes, la temperatura casi no cambia la ocupación.

## Cómo se lee el resultado

| Modelo | Coeficiente de la temperatura | Se lee |
|---|---|---|
| **Simple** (solo temperatura) | **−1.86** | «Por cada grado más, −1.86 puntos de ocupación» (parece un efecto fuerte) |
| **Con control** (temperatura + festivo + mes) | **−0.09**, p = **0.48** | «Por cada grado más, casi nada (−0.09), y no se distingue de cero» |

- **El efecto del calor casi desaparece:** era un espejismo del mes.
- **El efecto del festivo se mantiene:** +9 puntos con el modelo de control (era +16.3 sin controlar: parte de esa diferencia venía de que los festivos caen en meses de temporada alta).
- **Conclusión para el cliente:**
  - Los **festivos suben** la ocupación (efecto real, aguanta el control).
  - El **calor no la baja**: parece que sí solo porque coincide con la temporada.
- **Con datos observados no se puede *demostrar* causalidad.** Sí se pueden **descartar explicaciones simples** y decir cuál es más creíble. Demostrar una causa exige un experimento.

## Las palabras de la parte B

| Palabra | Qué significa |
|---|---|
| **Correlación (Pearson)** | Número de −1 a 1 que mide si dos variables suben o bajan juntas en línea recta |
| **Causalidad** | Que una cosa **provoque** la otra |
| **Variable de confusión** | Una variable (el mes) que influye en las otras dos y crea una relación falsa |
| **Regresión** | Un modelo que explica una variable (la ocupación) a partir de otras |
| **Fórmula `y ~ x`** | «y depende de x». Con `+` se añaden más variables |
| **Coeficiente (`.params`)** | Cuánto cambia `y` cuando esa variable sube una unidad |
| **`.pvalues`** | El p-valor de cada coeficiente |
| **`C(mes)`** | Trata el mes como una categoría (12 etiquetas) |
| **Intercept** | El valor de partida del modelo (cuando todo lo demás vale 0) |
| **`.fit()`** | Ajusta el modelo a los datos |
| **`groupby("mes")`** | Agrupa las filas por mes para calcular algo en cada grupo |

## Por qué se hace así

- **¿Por qué no basta con la correlación?**
  - Porque puede venir de una tercera variable. Si hacemos caso a −0.63, recomendaríamos «enfriar» el hotel: un consejo **inútil y caro**.
- **¿Por qué se añaden `festivo` y `C(mes)` juntos?**
  - Los festivos también cambian la ocupación. Al incluirlos, el modelo no atribuye al calor lo que es del festivo.
- **¿Por qué el coeficiente cambia de −1.86 a −0.09?**
  - Antes, la temperatura «cargaba» con el efecto del mes. Al tener en cuenta el mes, ese efecto se va a su sitio y a la temperatura le queda casi nada.
- **¿Qué se hace con un efecto que aguanta el control (el festivo)?**
  - Se confía más en él, pero sin prometer causalidad: puede haber otras variables que no hemos medido.

## 🚑 Si os atascáis

**Parte A · Contraste de hipótesis (ejercicio 1)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'festivos' is not defined` | No se ha ejecutado la celda 1.2 | Ejecutad las celdas desde el paso 0 hasta la 1.2, en orden |
| `❌ t de Student: estadístico` | Falta `equal_var=False` o se han puesto otras series | `stats.ttest_ind(festivos, normales, equal_var=False)` |
| `❌ t de Student: p-valor` | El p-valor sale muy grande: se han mezclado los grupos | Comprobad que `resultado` es el de la línea anterior y que `p_festivo = resultado.pvalue` (sin paréntesis) |
| `❌ t de Student: diferencia` | Se ha restado al revés o se ha usado la mediana | `festivos.mean() - normales.mean()`: festivos primero |
| `❌ t de Student: decisión` | Se ha escrito el texto `"True"` o se ha comparado con otro número | `p_festivo < 0.05`, sin comillas |
| `TypeError` o `AttributeError` que nombran `'ellipsis'` | Se ha quedado algún `...` sin sustituir | Cambiad los `...` por vuestro código |

**Parte B · Correlación y causalidad (ejercicio 5)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'modelo_simple' is not defined` | No se ha ejecutado la celda 3.2 | Ejecutad la celda 3.2 y después la 5.1 |
| `PatsyError` o `KeyError` al ajustar el modelo | Está mal escrito un nombre de columna en la fórmula | Los nombres son `ocupacion_pct`, `temperatura_c`, `festivo` y `mes`, todos en minúsculas |
| `❌ Confusión: modelo` | Falta `festivo` o `C(mes)` en la fórmula, o falta `.fit()` | `smf.ols("ocupacion_pct ~ temperatura_c + festivo + C(mes)", data=diaria).fit()` |
| `❌ Confusión: coeficiente simple` | Se ha usado el modelo de control en vez del simple | Es el del modelo de 3.2: `modelo_simple.params["temperatura_c"]` |
| `❌ Confusión: coeficiente con control` | Se ha usado el modelo simple en vez del de control | `modelo_control.params["temperatura_c"]` |
| `❌ Confusión: p-valor` | Se ha usado `.params` en vez de `.pvalues` | `modelo_control.pvalues["temperatura_c"]` |
| `KeyError: 'mes'` al hacer `groupby` | La celda 1.1 no se ha ejecutado entera (es la que crea la columna `mes`) | Ejecutad de nuevo la celda 1.1 |

**En cualquier parte**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'pd' is not defined` (o `np`, `stats`, `smf`…) | No habéis ejecutado la celda de las librerías, o Colab se reinició | Ejecutad las celdas **desde el paso 0** hasta la que falla, en orden |
| La comprobación da ❌ aunque creéis que está bien | Habéis corregido el código pero no habéis vuelto a ejecutar la celda del ejercicio | Ejecutad la celda del ejercicio **otra vez** y después la de comprobación |
| Sale un error al descargar el fichero (`URLError`, `FileNotFoundError`) | Sin internet o sin acceso a GitHub | Probad otra vez; si sigue, avisad al docente (puede dar el fichero `ocupacion_diaria.csv`) |
| Seguís sin verlo | A veces pasa | 1) Preguntad a vuestra pareja; 2) después, al docente; 3) **cuando hayáis terminado o agotado las pistas**, ejecutad la «Solución de referencia» del ejercicio. Primero intentadlo: lo que copiéis, tenéis que saber explicarlo |

## 🆘 Soluciones de referencia

- Ejecutad estas celdas **solo después de intentarlo**.
- Cada ejercicio tiene la suya.
- Si lo dejáis para después, ejecutadlas para poder seguir con el paso siguiente.

In [ ]:
#@title 🆘 Solución del ejercicio 1 · Contraste de hipótesis (ábrela solo después de intentarlo)
resultado = stats.ttest_ind(festivos, normales, equal_var=False)
p_festivo = resultado.pvalue
dif_medias = festivos.mean() - normales.mean()
rechazar_h0 = p_festivo < 0.05
print("Solución del ejercicio 1 aplicada: ahora ejecutad la celda 2.3")

In [ ]:
#@title 🆘 Solución del ejercicio 5 · Correlación y causalidad (ábrela solo después de intentarlo)
modelo_control = smf.ols("ocupacion_pct ~ temperatura_c + festivo + C(mes)", data=diaria).fit()
coef_temp_simple = modelo_simple.params["temperatura_c"]
coef_temp_control = modelo_control.params["temperatura_c"]
p_temp_control = modelo_control.pvalues["temperatura_c"]
print("Solución del ejercicio 5 aplicada: ahora ejecutad la celda 5.2")

## Para reflexionar (con vuestra pareja)

**Parte A · Contraste de hipótesis**

1. En 2.1 los grupos `c` y `d` tienen la misma media y sale p = 1.0. Cambiad `d` por `[80, 82, 78, 81, 79]` y volved a ejecutar: ¿qué pasa con el p-valor? ¿Por qué?
2. El p-valor de los festivos es minúsculo. ¿Quiere decir eso que la diferencia es **grande**? ¿Qué número os dice cuánto es?

**Parte B · Correlación y causalidad**

1. Con la correlación de −0.63 en la mano, ¿qué le recomendaría alguien que no hubiera mirado los meses al cliente? ¿Y vosotros?
2. El festivo aguanta el control: sigue subiendo +9 puntos. ¿Podéis estar **seguros** de que el festivo **causa** esa subida? ¿Qué os faltaría?

## Qué habéis aprendido

**Parte A · Contraste de hipótesis**

- Un **contraste** decide si una diferencia puede ser casualidad: se rechaza H0 si el **p-valor es menor que 0.05**.
- Con dos grupos se usa `stats.ttest_ind(a, b, equal_var=False)`.
- Un p-valor pequeño dice que la diferencia **existe**; la propia diferencia (16.3 puntos) dice **cuánto** es.

**Parte B · Correlación y causalidad**

- Correlación **no es** causalidad: una **variable de confusión** (el mes) puede crear una relación falsa.
- Un modelo de regresión con `smf.ols("y ~ x1 + x2 + C(cat)", data=…)` permite **tener en cuenta** otras variables.
- Al controlar el mes, el efecto del calor pasó de **−1.86 a −0.09**; el del festivo se mantuvo.